# Data Quality Audit System — Walkthrough

> A reproducible audit of messy business data: find the errors, classify and
> score them, clean them with documented decisions, and produce
> reporting-ready output.

A small B2B / online shop exports **sales, customer and product** data from
several systems. The monthly reports don't add up — IDs are missing or point
nowhere, sales are duplicated, prices go negative, dates arrive in three
formats, regions are spelled five ways. The question this notebook answers:

> **How reliable is this data — and which concrete errors are blocking
> correct reporting right now?**

This notebook tells the *story*. The same logic runs headless via
`run_pipeline.py`. Both call the exact same `src/` modules — nothing is
re-implemented here, only orchestrated and explained.

## 1. Setup

Load the `src/` modules and the single source of truth (`allowed_values.yaml`).

In [1]:
import sys
from pathlib import Path
from collections import Counter
import json
import yaml
import pandas as pd

# Make the project importable regardless of where the notebook runs from
ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from error_record import ErrorFactory, records_to_dataframe, KNOWN_CHECKS
from load_data import load_raw_data
from calculate_quality_score import (calculate_quality_score,
                                     count_total_rows, format_score_block)
from clean_data import clean
import validate_schema, validate_completeness, validate_uniqueness
import validate_relationships, validate_business_rules

LAYERS = [
    ("Schema",         validate_schema),
    ("Completeness",   validate_completeness),
    ("Uniqueness",     validate_uniqueness),
    ("Relationship",   validate_relationships),
    ("Business Rules", validate_business_rules),
]

config = yaml.safe_load((ROOT / "data/reference/allowed_values.yaml").read_text())
print(f"Config loaded — {len(KNOWN_CHECKS)} checks, fully mapped to "
      f"severity + cleaning action.")

Config loaded — 21 checks, fully mapped to severity + cleaning action.


## 2. The raw data

Four files. We load everything as text — raw stays raw, so the validators see values exactly as they appear in the files.

In [2]:
raw = load_raw_data(ROOT / "data/raw")
for name, df in raw.items():
    print(f"{name:<10} {df.shape[0]:>5} rows x {df.shape[1]} cols")
print(f"\nTotal rows checked: {count_total_rows(raw)}")

sales       2006 rows x 9 cols
customers    204 rows x 5 cols
products      53 rows x 5 cols
targets       60 rows x 3 cols

Total rows checked: 2323


A glimpse of what's wrong. Here are a few deliberately broken sales rows — missing IDs, a non-ISO date, an out-of-range discount, an unknown region:

In [3]:
manifest = json.loads((ROOT / "data/raw/injected_errors.json").read_text())
sample_refs = []
for cls in ("COM_001", "SCH_002", "BIZ_003", "BIZ_005", "REL_001"):
    hit = next(e for e in manifest["errors"] if e["error_class"] == cls)
    sample_refs.append(hit["row_ref"])

raw["sales"][raw["sales"]["sale_id"].isin(sample_refs)][
    ["sale_id", "sale_date", "customer_id", "product_id",
     "quantity", "unit_price", "discount", "region"]]

,sale_id,sale_date,customer_id,product_id,quantity,unit_price,discount,region
48,S-00049,27.03.2024,C-0020,P-022,9,69.29,0.25,Central
767,S-00765,2024-11-17,C-9900,P-006,6,273.84,0.25,South
1079,S-01076,2024-08-18,NaN,P-028,7,419.14,0.2,East
1112,S-01109,2024-09-30,C-0190,P-008,7,235.01,0.0,NORTH
1767,S-01763,2024-05-11,C-0033,P-031,6,514.33,-0.1,Central


## 3. Five validation layers

Each layer is **read-only** and emits the *same* error-record format. That
single contract is what makes everything downstream — scoring, cleaning,
reporting — a simple aggregation over one list.

In [4]:
factory = ErrorFactory(config)
validate_schema.assert_file_level_schema(raw, config)

raw_errors = []
for name, module in LAYERS:
    found = module.run_checks(raw, config, factory)
    raw_errors += found
    print(f"{name:<16} {len(found):>4} issues")
print(f"{'TOTAL':<16} {len(raw_errors):>4} issues")

Schema              9 issues
Completeness       43 issues
Uniqueness         13 issues
Relationship       26 issues
Business Rules     51 issues
TOTAL             142 issues


Business Rules     51 issues
TOTAL             142 issues


Every issue is one structured record — here are a few, across layers:

In [5]:
df_err = records_to_dataframe(raw_errors)
df_err.groupby("check_layer").head(1)[
    ["error_id", "check_id", "check_layer", "severity",
     "row_ref", "column", "invalid_value", "cleaning_action"]
].reset_index(drop=True)

,error_id,check_id,check_layer,severity,row_ref,column,invalid_value,cleaning_action
0,E00001,SCH_002,SCHEMA,MAJOR,S-00043,sale_date,2024/06/16,FIX
1,E00010,COM_001,COMPLETENESS,CRITICAL,S-00218,customer_id,<empty>,DROP
2,E00053,UNI_001,UNIQUENESS,MAJOR,S-00461,sale_id,S-00461,DROP
3,E00066,REL_003,RELATIONSHIP,MAJOR,S-00023,product_id,P-048,FLAG
4,E00092,BIZ_001,BUSINESS_RULE,CRITICAL,S-00019,quantity,-5,DROP


## 4. Trust: detection vs. the known error manifest

The demo generator injected every error **on purpose** and recorded each one.
We check detection in **both directions**: every injected error must be found,
and every detection must match an injection (no false positives on the clean
baseline). A validator that simply fired on everything would also reach
"100% found" — direction 2 is what rules that out.

In [6]:
injected = Counter((e["table"], e["row_ref"], e["error_class"])
                   for e in manifest["errors"])
detected = Counter((r.table, r.row_ref, r.check_id) for r in raw_errors)

matched = sum((injected & detected).values())
missed = sum((injected - detected).values())
unexpected = sum((detected - injected).values())
total = manifest["total_injected_errors"]

print(f"Known injected demo errors detected: {matched} / {total}")
print(f"Missed:     {missed}")
print(f"Unexpected: {unexpected}")
print()
print("Note: this refers only to the controlled, known demo errors —")
print("not to all possible data errors.")

Known injected demo errors detected: 142 / 142
Missed:     0
Unexpected: 0

Note: this refers only to the controlled, known demo errors —
not to all possible data errors.


## 5. Quality score — before cleaning

Weighted error points, normalised per 1,000 rows, subtracted from 100.
Weights live in the config (`CRITICAL 1.0 / MAJOR 0.4 / MINOR 0.1`), calibrated
so the score stays discriminating at realistic error rates instead of
collapsing to zero.

In [7]:
raw_score = calculate_quality_score(raw_errors, count_total_rows(raw), config)
print(format_score_block(raw_score, config, label="raw"))

Overall Quality Score (raw): 63/100        🔴

Critical Issues: 58     -> ACT
Major Issues:    66     -> CHECK
Minor Issues:    18     -> OBSERVE

Weighted error points: 86.2 (37.1 per 1000 rows, 2323 rows checked)

Traffic light thresholds (config): 🟢 >= 90   🟡 70-89   🔴 < 70


Where do the issues sit? Distribution by layer and the worst offenders:

In [8]:
top = pd.DataFrame(raw_score.top_error_classes(8),
                   columns=["check_id", "count"])
top["bar"] = top["count"].apply(lambda n: "#" * n)
top

,check_id,count,bar
0,COM_005,13,#############
1,REL_001,11,###########
2,BIZ_003,10,##########
3,SCH_002,9,#########
4,COM_001,8,########
5,COM_002,8,########
6,COM_004,8,########
7,REL_002,8,########


## 6. Cleaning — one documented decision per error

Every error class maps to exactly one action, defined in config:

- **FIX** — repair it (normalise the date, cap the discount, map the region)
- **DROP** — unrecoverable (missing FK, duplicate, negative amount)
- **FLAG** — keep it, but mark it (outlier, inactive product, missing optional)

A FIX can **escalate to a DROP** when a value can't be repaired — e.g. a region
with no known synonym ("Atlantis"). Nothing is silent.

In [9]:
result = clean(raw, raw_errors, config)

print(f"Dropped : {result.total_dropped}")
print(f"Fixed   : {result.total_fixed}")
print(f"Flagged : {result.total_flagged}")
print(f"FIX -> DROP escalations: {len(result.escalated_to_drop)}")
print()
print("Drop reasons:", dict(sorted(result.dropped_by_check.items())))

Dropped : 88
Fixed   : 29
Flagged : 35
FIX -> DROP escalations: 5

Drop reasons: {'BIZ_001': 8, 'BIZ_002': 7, 'BIZ_004': 6, 'BIZ_005': 3, 'BIZ_006': 2, 'COM_001': 8, 'COM_002': 8, 'COM_003': 6, 'COM_004': 8, 'REL_001': 11, 'REL_002': 8, 'UNI_001': 6, 'UNI_002': 4, 'UNI_003': 3}


The fix log records the original and corrected value for every repair — cleaning is never a black box:

In [10]:
pd.DataFrame([
    {"check": e.check_id, "column": e.column,
     "original": e.original_value, "fixed": e.fixed_value, "note": e.note}
    for e in result.fix_log
]).groupby("check").head(2).reset_index(drop=True)

,check,column,original,fixed,note
0,SCH_002,sale_date,2024/06/16,2024-06-16,date normalised to ISO 8601
1,SCH_002,sale_date,27.03.2024,2024-03-27,date normalised to ISO 8601
2,BIZ_006,sales_channel,Online,online,channel mapped
3,BIZ_005,region,west,West,region mapped
4,BIZ_006,sales_channel,STORE,retail,channel mapped
5,BIZ_005,region,EAST,East,region mapped
6,BIZ_003,discount,0.56,0.5,"discount clipped to [0, 0.5]"
7,BIZ_003,discount,0.67,0.5,"discount clipped to [0, 0.5]"


And the escalations — values that *looked* fixable but weren't:

In [11]:
pd.DataFrame([
    {"check": e.check_id, "original": e.original_value,
     "outcome": e.fixed_value, "note": e.note}
    for e in result.escalated_to_drop
])

,check,original,outcome,note
0,BIZ_005,Mordor,<dropped>,FIX impossible (no parse / no synonym) -> esca...
1,BIZ_006,carrier-pigeon,<dropped>,FIX impossible (no parse / no synonym) -> esca...
2,BIZ_005,Atlantis,<dropped>,FIX impossible (no parse / no synonym) -> esca...
3,BIZ_005,Springfield,<dropped>,FIX impossible (no parse / no synonym) -> esca...
4,BIZ_006,fax,<dropped>,FIX impossible (no parse / no synonym) -> esca...


## 7. Re-validate and score — after cleaning

We don't *subtract* the handled errors — we run the five validators again on
the cleaned data. That proves the cleaning actually worked. Whatever they
still find is genuine, documented residual uncertainty (the flagged rows).

In [12]:
factory2 = ErrorFactory(config)
validate_schema.assert_file_level_schema(result.cleaned, config)
clean_errors = []
for name, module in LAYERS:
    clean_errors += module.run_checks(result.cleaned, config, factory2)

clean_score = calculate_quality_score(
    clean_errors, count_total_rows(result.cleaned), config)

print(f"Residual issues: {len(clean_errors)}")
print(dict(sorted(Counter(r.check_id for r in clean_errors).items())))
print()
print("These are exactly the FLAG classes — kept on purpose, marked, not hidden.")

Residual issues: 25
{'BIZ_007': 5, 'COM_005': 13, 'REL_003': 7}

These are exactly the FLAG classes — kept on purpose, marked, not hidden.


## 8. Before / After

The business proof in one view.

In [13]:
comparison = pd.DataFrame({
    "metric": ["Quality score", "Critical", "Major", "Minor",
               "Total issues"],
    "before": [raw_score.score, raw_score.n_critical, raw_score.n_major,
               raw_score.n_minor, raw_score.total_errors],
    "after":  [clean_score.score, clean_score.n_critical,
               clean_score.n_major, clean_score.n_minor,
               clean_score.total_errors],
})
comparison

,metric,before,after
0,Quality score,63,98
1,Critical,58,0
2,Major,66,7
3,Minor,18,18
4,Total issues,142,25


In [14]:
print(f"Before: {raw_score.score}/100 {raw_score.emoji}"
      f"   ->   After: {clean_score.score}/100 {clean_score.emoji}")
print()
print(f"  before {'#' * raw_score.score}  ({raw_score.score})")
print(f"  after  {'#' * clean_score.score}  ({clean_score.score})")

Before: 63/100 🔴   ->   After: 98/100 🟢

  before ###############################################################  (63)
  after  ##################################################################################################  (98)


## 9. The clean master dataset

Sales joined with customers and products, with computed `gross_revenue`,
`net_revenue` and a `quality_flag` column. Every foreign key resolves (broken
ones were dropped), so revenue has no gaps.

In [15]:
master = result.master
print(f"{len(master)} reporting-ready rows, {master.shape[1]} columns")
master[["sale_id", "customer_name", "product_name", "quantity",
        "unit_price", "discount", "gross_revenue", "net_revenue",
        "quality_flag"]].head(6)

1925 reporting-ready rows, 18 columns


,sale_id,customer_name,product_name,quantity,unit_price,discount,gross_revenue,net_revenue,quality_flag
0,S-00001,Niklas Schmidt,Ergo Router 013,3,177.68,0.0,533.04,533.04,
1,S-00002,Tina Neumann,Pro Chair 003,3,254.16,0.05,762.48,724.36,
2,S-00003,Elena Neumann,Ergo Cable 027,3,367.82,0.1,1103.46,993.11,
3,S-00004,Ida Graf,Lite Dock 018,8,510.39,0.05,4083.12,3878.96,
4,S-00005,Rosa Eberhard,Max Tablet 030,4,24.41,0.15,97.64,82.99,
5,S-00006,Anna Conrad,Ergo Webcam 006,5,280.4,0.2,1402.00,1121.60,


The flagged rows — kept, but carrying their documented caveat:

In [16]:
flagged = master[master["quality_flag"].astype(str).str.len() > 0]
print(f"{len(flagged)} flagged rows")
flagged[["sale_id", "product_name", "quantity",
         "gross_revenue", "quality_flag"]].head(6)

35 flagged rows


,sale_id,product_name,quantity,gross_revenue,quality_flag
21,S-00023,Prime Tablet 048,8,1708.08,REL_003
47,S-00050,Prime Cable 023,3,1243.89,COM_005
118,S-00123,Compact Stand 050,1,131.97,REL_003
141,S-00149,Pro Chair 003,586,176321.54,BIZ_007
169,S-00177,Classic Stand 038,7,1779.12,COM_005
235,S-00248,Ergo Webcam 009,9,1898.10,COM_005


## 10. Conclusion

Starting from data that scored **63/100 🔴** — unusable for reporting — the
audit:

- found and classified **142** issues across five layers, verified
  **142 / 142** against a known error manifest,
- cleaned them with one documented FIX / DROP / FLAG decision each
  (88 dropped, 29 fixed, 35 flagged),
- and produced **1,925** reporting-ready rows scoring **98/100 🟢**.

The remaining 25 issues weren't swept away — they're **kept and flagged** as
honest residual uncertainty. That's the difference between data that *looks*
clean and data you can *trust*.

> *Code = measurement · Config = rules · Reports = interpretation*